# Preprocessing Step 3: Geospatial Join (Neukoelln)

This notebook spatially joins two exported datasets:
- neukoelln_green_roofs_decision_tree.shp
- reprocessed_buildings.shp (fallback: preprocessed_buildings.shp)

Goal: Create one complete, deduplicated output dataset for further analysis.

### Step 1: Resolve input paths
Find the project root, define input paths, and validate required files.

In [1]:
# Import required libraries
from pathlib import Path
import pandas as pd
import geopandas as gpd

In [2]:
# Resolve project root and define input paths

project_root = Path.cwd().resolve()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent

data_dir = project_root / "data"
step1_export_dir = data_dir / "exports" / "preprocessing_step1_clip_to_neuk"
step2_export_dir = data_dir / "exports" / "preprocessing_step2_sol_suit"

roofs_path = step1_export_dir / "neukoelln_greenroofs.shp"
buildings_path = step2_export_dir / "preprocessed_buildings_sol_suit.shp"

# Validate required input files
if not roofs_path.exists():
    raise FileNotFoundError(f"Missing file: {roofs_path}")
if not buildings_path.exists():
    raise FileNotFoundError(f"Missing file: {buildings_path}")

### Step 2: Import libraries and load layers
Import required libraries, then load the geospatial input layers.

In [3]:
# Load geospatial input layers
roofs_gdf = gpd.read_file(roofs_path)
buildings_gdf = gpd.read_file(buildings_path)

# Print quick input summary
print(f"Roofs: {len(roofs_gdf)} features")
print(f"Buildings: {len(buildings_gdf)} features")
print(f"Buildings file used: {buildings_path}")

Roofs: 53337 features
Buildings: 53337 features
Buildings file used: C:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\data\exports\preprocessing_step2_sol_suit\preprocessed_buildings_sol_suit.shp


### Step 3: Clean geometries and prepare attributes
Define helper functions, clean invalid geometries, align CRS, and prepare prefixed building attributes.

In [4]:
def clean_geometries(gdf: gpd.GeoDataFrame, name: str) -> gpd.GeoDataFrame:
    start = len(gdf)
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()

    invalid_mask = ~gdf.is_valid
    invalid_count = int(invalid_mask.sum())
    if invalid_count > 0:
        try:
            gdf.loc[invalid_mask, "geometry"] = gdf.loc[invalid_mask, "geometry"].make_valid()
        except Exception:
            gdf.loc[invalid_mask, "geometry"] = gdf.loc[invalid_mask, "geometry"].buffer(0)

    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty & gdf.is_valid].copy()
    removed = start - len(gdf)
    print(f"{name}: start={start}, invalid_before={invalid_count}, removed={removed}, remaining={len(gdf)}")
    return gdf

def prefix_attributes(gdf: gpd.GeoDataFrame, prefix: str) -> gpd.GeoDataFrame:
    rename_map = {c: f"{prefix}{c}" for c in gdf.columns if c != "geometry"}
    return gdf.rename(columns=rename_map)

In [5]:
# Clean geometries and align CRS
roofs_gdf = clean_geometries(roofs_gdf, "roofs")
buildings_gdf = clean_geometries(buildings_gdf, "buildings")

if buildings_gdf.crs != roofs_gdf.crs:
    buildings_gdf = buildings_gdf.to_crs(roofs_gdf.crs)

# Prefix building attributes to avoid name collisions during join
buildings_join = prefix_attributes(buildings_gdf, "bldg_")

roofs: start=53337, invalid_before=0, removed=0, remaining=53337
buildings: start=53337, invalid_before=0, removed=0, remaining=53337


In [6]:
# Spatial join roofs with building attributes without intersects-duplicates
roofs_base = roofs_gdf.reset_index(drop=True).copy()
roofs_base["roof_row_id"] = roofs_base.index

# Use representative points for robust 1:1 building attribution
roof_points = roofs_base[["roof_row_id", "geometry"]].copy()
roof_points = roof_points.set_geometry(roof_points.geometry.representative_point())

# Join points to buildings (within avoids neighboring polygon touches)
point_join = gpd.sjoin(
    roof_points,
    buildings_join,
    how="left",
    predicate="within",
)

# If overlaps still exist, keep first match per roof
point_join = point_join.sort_values(["roof_row_id", "index_right"], na_position="last")
point_join = point_join.drop_duplicates(subset=["roof_row_id"], keep="first")

# Keep only joined building attributes and merge back to original roof polygons
bldg_attr_cols = [c for c in buildings_join.columns if c != "geometry"]
join_cols = ["roof_row_id"] + [c for c in bldg_attr_cols if c in point_join.columns]
roof_bldg_attrs = point_join[join_cols].copy()

joined_full = roofs_base.merge(roof_bldg_attrs, on="roof_row_id", how="left")
joined_full = joined_full.drop(columns=["roof_row_id"], errors="ignore")

print("Joined columns:")
print(joined_full.columns)
print(f"Joined features (expected ~= roofs): {len(joined_full)}")

Joined columns:
Index(['target_0_1', 'roof_area_', 'gruen20_m2', 'gruen20_p', 'gint20_m2',
       'gex20_m2', 'geometry', 'bldg_index', 'bldg_ex_int', 'bldg_gebaeudefu',
       'bldg_bauweise', 'bldg_ist_denkma', 'bldg_anzahl_unt',
       'bldg_anzahl_obe', 'bldg_verschattu', 'bldg_verschat_1',
       'bldg_verschat_2', 'bldg_suitable_a', 'bldg_suitabilit'],
      dtype='str')
Joined features (expected ~= roofs): 53337


In [7]:
# Safety check for duplicates after point-based join
joined_full_dedup = joined_full.copy()
joined_full_dedup["_geom_key"] = joined_full_dedup.geometry.apply(
    lambda g: g.wkb_hex if g is not None else None
)

before = len(joined_full_dedup)
joined_full_dedup = joined_full_dedup.drop_duplicates(subset=["_geom_key"], keep="first")
after = len(joined_full_dedup)
joined_full_dedup = joined_full_dedup.drop(columns=["_geom_key"], errors="ignore")

print(f"Joined features before safety dedup: {before}")
print(f"Joined features after safety dedup: {after}")
print(f"Removed duplicates: {before - after}")

Joined features before safety dedup: 53337
Joined features after safety dedup: 53337
Removed duplicates: 0


In [8]:
# Configure output paths in dedicated export subfolder
output_dir = data_dir / "exports" / "preprocessing_step3_geosp_join"
output_dir.mkdir(parents=True, exist_ok=True)

output_csv = output_dir / "geospatial_join_complete_neukoelln.csv"
output_gpkg = output_dir / "geospatial_join_complete_neukoelln.gpkg"
output_shp = output_dir / "geospatial_join_complete_neukoelln.shp"

# Export CSV without geometry
joined_full_dedup.drop(columns=["geometry"], errors="ignore").to_csv(output_csv, index=False)

# Export full layer to GPKG
joined_full_dedup.to_file(output_gpkg, layer="geospatial_join_complete_dedup", driver="GPKG")

# Export SHP only for Polygon/MultiPolygon geometries
polygon_types = {"Polygon", "MultiPolygon"}
joined_full_poly = joined_full_dedup[joined_full_dedup.geometry.geom_type.isin(polygon_types)].copy()
if len(joined_full_poly) > 0:
    joined_full_poly.to_file(output_shp)
    print(f"SHP saved: {output_shp} ({len(joined_full_poly)} features)")
else:
    print("No SHP exported: no polygon geometries found.")

print(f"CSV saved: {output_csv}")
print(f"GPKG saved: {output_gpkg} ({len(joined_full_dedup)} features)")

C:\Users\elbma\AppData\Local\Temp\ipykernel_30300\2385871431.py:19: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  joined_full_poly.to_file(output_shp)
c:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'bldg_ex_int' to 'bldg_ex_in'
  ogr_write(
c:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'bldg_gebaeudefu' to 'bldg_gebae'
  ogr_write(
c:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'bldg_bauweise' to 'bldg_bauwe'
  ogr_write(
c:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\.venv\Lib\site-packages\pyogrio\raw.py:733: Runtime

SHP saved: C:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\data\exports\preprocessing_step3_geosp_join\geospatial_join_complete_neukoelln.shp (53337 features)
CSV saved: C:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\data\exports\preprocessing_step3_geosp_join\geospatial_join_complete_neukoelln.csv
GPKG saved: C:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\data\exports\preprocessing_step3_geosp_join\geospatial_join_complete_neukoelln.gpkg (53337 features)
